# House Price Prediction

A reproducible tabular regression workflow: EDA → leakage-safe preprocessing → baseline → cross-validation → tuning → final holdout evaluation.

## 1. Imports and configuration

The notebook is for analysis and communication. Reusable training and inference logic lives under `src/house_price_ml/`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))

from house_price_ml.data import load_dataset, make_train_test_split, split_features_target
from house_price_ml.evaluation import regression_metrics
from house_price_ml.models import get_candidate_models, get_tuning_grid
from house_price_ml.pipeline import build_model_pipeline
from sklearn.model_selection import GridSearchCV, KFold, cross_validate

RANDOM_STATE = 42
DATA_PATH = ROOT / "data" / "housing.csv"
TARGET_COLUMN = "median_house_value"

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:.3f}")
sns.set_theme(style="darkgrid")


## 2. Load and inspect the dataset

In [ ]:
housing = load_dataset(DATA_PATH)
print(f"Shape: {housing.shape}")
housing.head()


In [ ]:
housing.info()
housing.describe(include="all").T


## 3. Data-quality checks

Before modeling, verify missing values, duplicate rows, categorical cardinality, and suspicious target distributions.

In [ ]:
print("Missing values:")
print(housing.isna().sum().sort_values(ascending=False))

print("\nDuplicate rows:", housing.duplicated().sum())

for column in housing.select_dtypes(exclude="number").columns:
    print(f"\n{column}:")
    print(housing[column].value_counts().head(20))

print("\nMost frequent target values:")
print(housing[TARGET_COLUMN].value_counts().head(20))


## 4. Exploratory analysis

In [ ]:
numeric_columns = housing.select_dtypes(include="number").columns
housing[numeric_columns].hist(figsize=(14, 10), bins=40)
plt.tight_layout()
plt.show()


In [ ]:
correlation = housing[numeric_columns].corr(numeric_only=True)[TARGET_COLUMN].sort_values(ascending=False)
correlation


In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=housing, x="median_income", y=TARGET_COLUMN, alpha=0.25)
plt.title("Median Income vs. Median House Value")
plt.show()


## 5. Split before fitting preprocessing

The test set is isolated before any imputation, scaling, encoding, cross-validation, or tuning. This is the key leakage-prevention boundary.

In [ ]:
X, y = split_features_target(housing, TARGET_COLUMN)
X_train, X_test, y_train, y_test = make_train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)


## 6. Build the reusable preprocessing pipeline

Numeric features receive median imputation and scaling. Non-numeric features receive most-frequent imputation and one-hot encoding. `handle_unknown="ignore"` keeps inference robust to unseen categories.

In [ ]:
from sklearn.linear_model import LinearRegression

baseline = build_model_pipeline(X_train, LinearRegression(), dense_output=True)
baseline.fit(X_train, y_train)

baseline_predictions = baseline.predict(X_test)
regression_metrics(y_test, baseline_predictions)


## 7. Cross-validate candidate models

Model comparison happens on training data only. RMSE is the primary selection metric.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "r2": "r2",
}

rows = []
for name, estimator in get_candidate_models(RANDOM_STATE).items():
    pipeline = build_model_pipeline(X_train, estimator, dense_output=True)
    result = cross_validate(
        pipeline, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1
    )
    rows.append({
        "model": name,
        "rmse_mean": -result["test_rmse"].mean(),
        "rmse_std": result["test_rmse"].std(),
        "mae_mean": -result["test_mae"].mean(),
        "r2_mean": result["test_r2"].mean(),
    })

cv_results = pd.DataFrame(rows).sort_values("rmse_mean")
cv_results


## 8. Tune the selected model

In [ ]:
selected_name = cv_results.iloc[0]["model"]
selected_estimator = get_candidate_models(RANDOM_STATE)[selected_name]
selected_pipeline = build_model_pipeline(X_train, selected_estimator, dense_output=True)
param_grid = get_tuning_grid().get(selected_name, {})

if param_grid:
    search = GridSearchCV(
        selected_pipeline,
        param_grid=param_grid,
        scoring="neg_root_mean_squared_error",
        cv=cv,
        n_jobs=-1,
        refit=True,
    )
    search.fit(X_train, y_train)
    final_model = search.best_estimator_
    print("Selected model:", selected_name)
    print("Best parameters:", search.best_params_)
    print("Best CV RMSE:", -search.best_score_)
else:
    final_model = selected_pipeline.fit(X_train, y_train)


## 9. Final evaluation

The holdout test set is used only now, after model selection and tuning.

In [ ]:
final_predictions = final_model.predict(X_test)
final_metrics = regression_metrics(y_test, final_predictions)
final_metrics


## 10. Interpretation and next steps

Record the final RMSE, MAE, and R² in project documentation only after this final evaluation. Useful next steps are residual analysis, feature engineering, explainability, and experiment tracking. Keep the test set untouched while experimenting.